<a href="https://colab.research.google.com/github/thinus283-ux/LR/blob/main/Global_parameter_convergence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install dynesty

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.9/102.9 kB 2.5 MB/s eta 0:00:00


import dynesty
import numpy as np
import matplotlib.pyplot as plt

# 1. OPTIMIZED CONSTRAINED PHYSICS
def log_likelihood_calibrated(theta):
    alpha, omega_m = theta
    
    # Enforce physical bounds
    if not (0.1 < omega_m < 0.5):
        return -np.inf
    
    # Calibrated LR Scaling Constants (Power-law exponents)
    h0_base = 67.4
    growth_factor = 1.0 - (0.18 * (alpha ** 1.5))
    h0_theory = h0_base * (1.0 + (alpha ** 0.8) * 0.125)
    s8_theory = 0.834 * np.sqrt(omega_m / 0.3) * growth_factor
    
    # Chi-Squared against observational constraints (SH0ES + KiDS)
    chi2 = ((h0_theory - 73.04)**2 / (1.04**2)) + \
           ((s8_theory - 0.759)**2 / (0.024**2))
    
    return -0.5 * chi2

# 2. MATCHING PRIOR TRANSFORM
def ptform_inflated(u):
    # Stressed Prior Space: Alpha [0, 6] (Forces the stress test)
    # Omega_m [0.1, 0.5] (Physical range)
    alpha = u[0] * 6.0          
    omega_m = 0.1 + u[1] * 0.4  
    return np.array([alpha, omega_m])

# 3. EXECUTION
ndim = 2
sampler = dynesty.NestedSampler(log_likelihood_calibrated, ptform_inflated, ndim, nlive=500)
print("Running Prior Inflation Stress Test (Optimized)...")
sampler.run_nested(dlogz=0.1)

# 4. RESULTS
results = sampler.results
median_alpha = np.median(results.samples[:, 0])
median_omega = np.median(results.samples[:, 1])

print(f"\n=== Final Stress Test Results ===")
print(f"Baseline Target  : 0.630")
print(f"Inflated Alpha   : {median_alpha:.3f}")
print(f"Inflated Omega_m : {median_omega:.3f}")

# 5. VISUALIZATION
plt.figure(figsize=(8, 5))
plt.hist(results.samples[:, 0], bins=50, color='skyblue', edgecolor='black', alpha=0.7)
plt.axvline(x=0.63, color='red', linestyle='--', label='Target: 0.63')
plt.title("Posterior Distribution: Prior Inflation Stress Test")
plt.xlabel("Alpha")
plt.ylabel("Frequency")
plt.legend()
plt.show()